In [1]:
import pandas as pd
import numpy as np
from abiasales.data_generation import *
from abiasales.cleaning import remove_outliers
from abiasales.power import calc_power_table
from abiasales.results import calc_exp_results
from abiasales.simulation import construct_exp_configs, run_aa_test_simulation, run_ab_test_simulation

In [2]:
np.random.seed(42)  # synthetic data and simulations are reproducible
pd.options.display.float_format = '{:,.2f}'.format

# DATA

In [3]:
df_a = create_test_data(uplift=0, n_users=1000000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.1, n_users=1000000)
df_b['exp_group'] = 'B'

df = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column         Dtype  
---  ------         -----  
 0   uid            int64  
 1   platform       str    
 2   country        str    
 3   strata         str    
 4   views          int64  
 5   clicks         int64  
 6   payer_flag     int64  
 7   purchases      int64  
 8   views_cov      float64
 9   clicks_cov     float64
 10  purchases_cov  float64
 11  exp_group      str    
dtypes: float64(3), int64(5), str(4)
memory usage: 183.1 MB


In [5]:
STRATS_COLS = ['platform', 'country']
FEATURE_COLS = ['views_cov', 'clicks_cov', 'purchases_cov']

In [6]:
df.groupby(['exp_group']).agg(
    users=('uid', 'count'),
    views=('views', 'mean'),
    clicks=('clicks', 'mean'),
    payers=('payer_flag', 'mean'),
    purchases=('purchases', 'mean'),
    views_cov=('views_cov', 'mean'),
    clicks_cov=('clicks_cov', 'mean'),
    purchases_cov=('purchases_cov', 'mean')
)

,users,views,clicks,payers,purchases,views_cov,clicks_cov,purchases_cov
exp_group,,,,,,,,
A,1000000,13.04,1.72,0.25,2.08,13.04,1.71,2.07
B,1000000,13.05,1.89,0.25,2.28,13.04,1.72,2.07


Check correlation of metric and covariates

In [7]:
from scipy.stats import pearsonr

print('Correlation')
for f in FEATURE_COLS:
    print(pearsonr(df['purchases'], df[f])[0])

Correlation
0.005715366186763351
0.014086023167271634
0.29910945764539587


Removing outliers

In [8]:
df = remove_outliers(df, metrics=FEATURE_COLS, contamination=0.001)

# SAMPLE SIZE

In [9]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    alpha=[0.05], power=[0.8], uplift=[0.01, 0.03, 0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1.0%,759304
3.0%,84368
5.0%,30374


In [10]:
# WITH STRATIFICATION
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,26741


In [11]:
# WITH CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,27672


In [12]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,23086


# MDE

MDE calculation accounting for given sample size

In [13]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[int(len(df) / 2)], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
999000,0.9%


In [14]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], sample_size=[int(len(df) / 2)], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
999000,0.8%


# RESULTS

In [15]:
df_a = create_test_data(uplift=0, n_users=20000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.05, n_users=20000)
df_b['exp_group'] = 'B'

df_results = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

Calculating experiment results

List of metric config parameters:
- **num** - name of the column containing the numerator of the metric.
- **den** - name of the column containing the denominator of the metric. Defaults to None and can be omitted.
- **weight_method** - type of user weighting. Default is 'uniform'. Possible values: 'uniform', 'size', 'sqrt', 'intra_corr'.
- **apply_linearization** - whether to linearize a ratio metric. Default is False. The linearized metric is expressed in ratio units of the control group, so means and uplifts are comparable with other methods.
- **use_delta_method** - whether to use the delta method for variance estimation (relevant only for ratio metrics). Requires weight_method='size', because the delta method estimates the ratio of sums.
- **stat_method** - statistical test used to calculate the p-value. Default is 't_test' (Welch's t-test). Possible values: 'z_test', 't_test', 'proportion' (for binary 0/1 metrics only), 'bootstrap'.
- **uplift_type** - 'rel' (relative, default) or 'abs' (absolute) uplift.
- **var_reduction_method** - variance reduction method. Default is None. Possible values: None, 'cuped', 'cupac'.
- **var_reduction_covariates** - list of covariates for CUPED/CUPAC. Required if var_reduction_method is set.
- **var_reduction_params** - parameters of the CatBoost model for CUPAC, e.g. {'iterations': 100}. Defaults to None.
- **handle_outliers** - how to handle outliers. Default is None. Possible values: remove (IsolationForest), cap (capping values).
- **outliers_contamination** - threshold to define an outlier. Default is 0.001. In case of handle_outliers="remove" – top outliers_contamination % of samples, in case of handle_outliers="cap" (capping values) – top quantile level of capping.
- **outliers_cols** – columns to detect outliers, relevant only for handle_outliers="remove". Defaults to the numerator and denominator columns.

Incompatible combinations raise a ValueError: e.g. the delta method without 'size' weights, the proportion test with linearization, variance reduction or stratification, the bootstrap with stratification.

Stratification is configured at the calc_exp_results function level and can be overridden in a metric config:
- **use_stratification** - whether to apply stratification. Defaults to False.
- **strats_cols** - list of columns used for stratification. Defaults to None. With use_stratification = True, CUPED/CUPAC models are also fitted per stratum.

The function returns two p-value values:
- **pvalue_init** – the initial p-value obtained from the statistical test.
- **pvalue** – the p-value adjusted for multiple hypothesis testing.

The returned table is numeric: **uplift** and **confint** are fractions for relative uplift (0.05 means +5%). The displayed table is formatted with `format_results(results)`.

In [16]:
# Target metrics
metrics = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test'
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test'
    }
}

# Same metrics but with CUPED and outliers handling
metrics_cuped = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test',
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'remove',
        'outliers_cols': FEATURE_COLS,
        'outliers_contamination': 0.001
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test',
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'remove',
        'outliers_cols': FEATURE_COLS,
        'outliers_contamination': 0.001
    }
}

In [17]:
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics
)

,metric,group_1,group_2,mean_1,mean_2,uplift_type,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.085400,2.199300,rel,5.46%,"(0.99%, 10.13%)",0.016051,0.016051
1,CTR,A,B,0.130498,0.140176,rel,7.42%,"(5.68%, 9.18%)",0.000000,0.000000


In [18]:
# WITH STRATIFICATION AND CUPED
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics_cuped,
    use_stratification=True, strats_cols=STRATS_COLS
)

,metric,group_1,group_2,mean_1,mean_2,uplift_type,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.084818,2.190167,rel,5.05%,"(1.16%, 9.10%)",0.010594,0.010594
1,CTR,A,B,0.128736,0.137956,rel,7.16%,"(5.00%, 9.38%)",0.000000,0.000000


# SIMULATION

In this section, we will simulate A/A and A/B tests to verify that the p-value properly controls the Type I error rate and to compare the power of different statistical tests.

Important note: There are multiple ways to generate synthetic uplift for A/B test simulations, and the results can vary significantly depending on the chosen approach.
The library implements the following options (`metric_type` argument of `run_ab_test_simulation`):

- **proportion** – the simplest option, where uplift can be generated directly (suitable for proportion-type metrics).
- **value** – used for metrics such as profit, purchases, etc. A normal random variable with a mean equal to the absolute uplift value is added. This is a generally applicable and versatile option.
- **ratio** – the most complex case. The library adds a normal random component to all units where the metric value is not zero. However, this approach has certain drawbacks — for instance, it interacts poorly with CUPED.

Each of the functions **`run_aa_test_simulation`** and **`run_ab_test_simulation`** takes a **`configs`** argument — a list of dictionaries, where each dictionary defines the metric configuration and the statistical methods to be used.  

Each dictionary has the following **structure**:

---

- **`num`** *(str)*  
  Name of the column with the metric’s numerator (e.g., `"purchases"`).

- **`den`** *(str | None)*  
  Name of the column with the metric’s denominator.  
  If `None`, assumes a value of 1.

- **`weight_method`** *(str)*  
  Unit-level weighting scheme to apply.  
  Allowed:  
  - `'uniform'`: all users weight = 1  
  - `'size'`: weights equal to the metric “size” (e.g., denominator or exposure)  
  - `'sqrt'`: weights are √(size)  
  - `'intra_corr'`: correlation-aware weights (accounts for within-user correlation)  
  Default: `'uniform'`.

- **`apply_linearization`** *(bool)*  
  Whether to linearize a ratio metric.  
  Default: `False`.

- **`use_delta_method`** *(bool)*  
  Whether to use the delta method for a ratio metric. Requires `weight_method='size'`.  
  Default: `False`.

- **`stat_method`** *(str)*  
  Statistical test to compute the p-value.  
  Allowed: `'proportion'`, `'z_test'`, `'t_test'`, `'bootstrap'`.  
  Use `'proportion'` only for binary (0/1) metrics.  
  Default: `'t_test'`.

- **`use_stratification`** *(bool)*  
  Whether to apply stratification.  
  Default: `False`.

- **`strats_cols`** *(list[str] | None)*  
  Columns to use as strata keys (e.g., `device`, `country`).  
  With `use_stratification=True`, CUPED/CUPAC models are also fitted per stratum.  
  Default: `None`.

- **`var_reduction_method`** *(str)*  
  Variance reduction method.  
  Allowed: `None`, `'cuped'`, `'cupac'`.  
  Default: `None`.

- **`var_reduction_covariates`** *(list[str] | None)*  
  Covariates for CUPED/CUPAC.  
  Ignored if `var_reduction_method=None`.  
  Default: `None`.

- **`var_reduction_params`** *(dict | None)*  
  Parameters of the CatBoost model for CUPAC (e.g., `{'iterations': 100}`).  
  Default: `None`.

- **`handle_outliers`** *(str | None)*  
  Strategy for outliers.  
  Allowed:  
  - `None`  
  - `'remove'`: drop outliers using IsolationForest  
  - `'cap'`: cap metric at quantile threshold  
  Default: `None`.

- **`outliers_contamination`** *(float | None)*  
  Expected outlier fraction (e.g., `0.001` = 0.1%).  
  Used to set cutoffs for `'remove'`/`'cap'`.  
  Ignored if `handle_outliers=None`.  
  Default: `0.001`.

- **`outliers_cols`** *(list[str])*  
  Columns on which to detect/handle outliers (e.g., `exposure`, `spend`, `price`).  
  Ignored if `handle_outliers != 'remove'`. Defaults to the numerator and denominator columns.

- **`uplift_type`** *(str)*  
  `'rel'` or `'abs'`.  
  Default: `'rel'`.

Configs are usually built with `construct_exp_configs`, which skips incompatible combinations of methods.

---

**Example:**

```python
{
    'num': 'purchases',
    'weight_method': 'uniform',
    'stat_method': 'z_test',
    'use_stratification': False,
    'strats_cols': None,
    'var_reduction_method': None,
    'var_reduction_covariates': ['purchases_cov'],    # ignored because var_reduction_method=None
    'handle_outliers': None,
    'outliers_contamination': 0.001,                  # ignored because handle_outliers=None
    'outliers_cols': ['purchases_cov']                # ignored because handle_outliers=None
}
```

In [19]:
df_sim = df[df['exp_group']=='A'] # control group sample
n_exp = 1000 # number of experiments in simulations

## PROPORTION

In [20]:
sample_size = 1000

pt, pw = calc_power_table(
    df=df_sim,
    metric_num='payer_flag',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1000,21.7%


Compare different statistical tests

In [21]:
configs = construct_exp_configs(
    metric_num='payer_flag',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['proportion', 'z_test', 't_test'], # 'proportion', 'z_test', 't_test', 'bootstrap'
    use_stratification=[False], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

First, run synthetic A/A tests.

In [22]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'proportion', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 532.31it/s]


Running A/A for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 550.41it/s]


Running A/A for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 573.39it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,proportion,False,None,None,None,0.487132,True
1,z_test,False,None,None,None,0.515232,True
2,t_test,False,None,None,None,0.493578,True


All tests are valid.

Second, run A/B tests in order to compare powers of tests

In [23]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='proportion',
    uplift=model_uplift
)

Running A/B for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'proportion', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:24<00:00, 41.14it/s]


Running A/B for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:24<00:00, 41.31it/s]


Running A/B for config: {'num': 'payer_flag', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:24<00:00, 41.45it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,proportion,False,None,None,None,0.807000
1,z_test,False,None,None,None,0.785000
2,t_test,False,None,None,None,0.776000


An illustrative example – all statistical tests show roughly the same power: 0.78–0.81, the difference is within the simulation noise (about ±0.025 for 1000 experiments). For conversion metrics, even with relatively small sample sizes (e.g., 1000 in this example), there is no practical difference in which test is used.

## VALUE

In [24]:
sample_size = 1000

pt, pw = calc_power_table(
    df=df_sim,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1000,27.8%


First, compare 3 main statistical tests

In [25]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['z_test', 't_test', 'bootstrap'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [26]:
configs

[{'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 'z_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': None,
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'var_reduction_params': None,
  'handle_outliers': None,
  'outliers_contamination': None,
  'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov']},
 {'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': None,
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'var_reduction_params': None,
  'handle_outliers': None,
  'outliers_contamination': None,
  'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov']},
 {'num': '

In [27]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 565.14it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 568.42it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'bootstrap', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:53<00:00, 18.78it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,z_test,False,None,None,None,0.497837,True
1,t_test,False,None,None,None,0.511304,True
2,bootstrap,False,None,None,None,0.513273,True


All the tests are valid.

Compare powers:

In [28]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'z_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 262.10it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 256.16it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 'bootstrap', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:55<00:00, 18.08it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,z_test,False,None,None,None,0.785000
1,t_test,False,None,None,None,0.796000
2,bootstrap,False,None,None,None,0.779000


Another illustrative case: there is no difference between the z-test and the t-test, since asymptotically they are equivalent, and 1000 users per group is already enough for that. In practice, sample sizes are usually much larger, so the choice of statistical test makes no real difference.<br>
The bootstrap power (0.78) is also within the simulation noise. On highly skewed distributions it can be lower.

Check how stratification works

In [29]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False, True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [30]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:01<00:00, 565.09it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 304.77it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,t_test,False,None,None,None,0.500447,True
1,t_test,True,None,None,None,0.503700,True


In [31]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 266.07it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:05<00:00, 189.89it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,t_test,False,None,None,None,0.783000
1,t_test,True,None,None,None,0.823000


We observe roughly a 0.04 increase in power due to stratification (0.78 -> 0.82).<br>
This example is based on synthetic data, which is why the gain is relatively large.
In practice, the improvement is likely to be smaller.
However, adding stratification is almost always a cost-free operation, so there’s little reason to ignore it.

Try to add CUPED and CUPAC

In [32]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None, 'cuped', 'cupac'], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [33]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:03<00:00, 312.44it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:08<00:00, 123.71it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cupac', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [09:42<00:00,  1.72it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,t_test,True,nan,None,None,0.499791,True
1,t_test,True,cuped,None,None,0.509987,True
2,t_test,True,cupac,None,None,0.500529,True


All options are valid. CUPAC models are cross-fitted: each user is adjusted by models that have not seen them. Otherwise CatBoost overfits to the experiment data and the variance is underestimated.

Compare powers

In [34]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:05<00:00, 190.09it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:10<00:00, 98.01it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cupac', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [09:58<00:00,  1.67it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,t_test,True,nan,None,None,0.831000
1,t_test,True,cuped,None,None,0.883000
2,t_test,True,cupac,None,None,0.824000


CUPED provides a clear improvement in statistical power (0.83 -> 0.88).<br>
CUPAC gives no gain here: the covariates are linearly related to the metric, and with ~250 users per stratum CatBoost can't learn more than linear regression. CUPAC is worth trying with nonlinear dependencies and larger samples.

Compare handling outliers strategies

In [35]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=['cuped'], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None, 'remove', 'cap'], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [36]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:08<00:00, 123.18it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:30<00:00, 11.06it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'cap', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:08<00:00, 117.80it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,t_test,True,cuped,nan,nan,0.504718,True
1,t_test,True,cuped,remove,0.001000,0.502837,True
2,t_test,True,cuped,cap,0.001000,0.498367,True


In [37]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:10<00:00, 98.54it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:32<00:00, 10.82it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'cap', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:10<00:00, 93.13it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,t_test,True,cuped,nan,nan,0.867000
1,t_test,True,cuped,remove,0.001000,0.882000
2,t_test,True,cuped,cap,0.001000,0.902000


All options are valid. Capping the metric at the 99.9% quantile gives the highest power (0.90 vs 0.87 without outlier handling), removing outliers by pre-experiment features is in between (0.88).<br>
Note that capping changes the metric itself: the means of capped values are compared.

Compare outliers contamination levels

In [38]:
configs = construct_exp_configs(
    metric_num='purchases',
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=['cuped'], # None, 'cuped'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=['remove'], # None, 'remove', 'cap',
    outliers_contamination=[0.001, 0.005, 0.01], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [39]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:30<00:00, 11.06it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.005, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:30<00:00, 11.03it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.01, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:30<00:00, 11.07it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,t_test,True,cuped,remove,0.001000,0.510308,True
1,t_test,True,cuped,remove,0.005000,0.501362,True
2,t_test,True,cuped,remove,0.010000,0.507412,True


In [40]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.001, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:32<00:00, 10.82it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.005, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:32<00:00, 10.77it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'var_reduction_params': None, 'handle_outliers': 'remove', 'outliers_contamination': 0.01, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [01:32<00:00, 10.80it/s]


,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,t_test,True,cuped,remove,0.001000,0.857000
1,t_test,True,cuped,remove,0.005000,0.876000
2,t_test,True,cuped,remove,0.010000,0.884000


Removing more outliers gives a little bit better statistical power: 0.86 for 0.1%, 0.88 for 0.5% and 1%.

Overall, best setup is:
- t-test
- Use stratification
- Use CUPED
- Cap the metric at the 99.9% quantile, or remove 0.5–1% of outliers by pre-experiment features if the metric must stay uncapped

It increases statistical power from ~0.79 for the default t-test without anything to ~0.90, i.e. by about 14%.

## RATIO

In [41]:
sample_size = 10000

pt, pw = calc_power_table(
    df=df_sim,
    metric_num='clicks',
    metric_den='views',
    weight_method='size',
    use_delta_method=True,
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
10000,3.3%


Compare two weighting methods, the use of linearization, and the delta method.

In [42]:
configs = construct_exp_configs(
    metric_num='clicks',
    metric_den='views',
    weight_methods=['uniform', 'size'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    apply_linearization=[False, True], # False, True
    use_delta_method=[False, True], # False, True
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

Filter out obviously uninteresting combinations

In [43]:
configs = [
    s for s in configs
    if
        not (s['weight_method']=='size' and s['apply_linearization'])
        and not (s['apply_linearization'] and s['weight_method']!='uniform')
        and not (not s['apply_linearization'] and s['weight_method']=='uniform')
]

Simulate A/A-tests

In [44]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'uniform', 'apply_linearization': True, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:06<00:00, 152.82it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:06<00:00, 159.79it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:06<00:00, 160.39it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,True,False,t_test,False,None,None,None,0.502021,True
1,size,False,False,t_test,False,None,None,None,0.489759,True
2,size,False,True,t_test,False,None,None,None,0.498129,True


All options are valid. With size weights the standard deviation is estimated with the delta method in both cases, so `use_delta_method` makes no difference here.

Compare powers:

In [45]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='ratio',
    uplift=model_uplift
)

Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'uniform', 'apply_linearization': True, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:08<00:00, 122.90it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:07<00:00, 127.53it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:07<00:00, 129.90it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,uniform,True,False,t_test,False,None,None,None,0.802000
1,size,False,False,t_test,False,None,None,None,0.788000
2,size,False,True,t_test,False,None,None,None,0.778000


All options perform roughly the same (0.78–0.80), so we’ll keep only size + delta_method as the default.

Now let’s try adding stratification.

In [46]:
configs = construct_exp_configs(
    metric_num='clicks',
    metric_den='views',
    weight_methods=['size'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    apply_linearization=[False], # False, True
    use_delta_method=[True], # False, True
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False, True], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [47]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:06<00:00, 158.39it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:11<00:00, 89.75it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,size,False,True,t_test,False,None,None,None,0.501249,True
1,size,False,True,t_test,True,None,None,None,0.487671,True


In [48]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='ratio',
    uplift=model_uplift
)

Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:07<00:00, 126.07it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': True, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:12<00:00, 80.82it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,size,False,True,t_test,False,None,None,None,0.790000
1,size,False,True,t_test,True,None,None,None,0.854000


Stratification provides a substantial increase in statistical power (0.79 -> 0.85).

Now let’s compare different weighting methods.

In [49]:
configs = construct_exp_configs(
    metric_num='clicks',
    metric_den='views',
    weight_methods=['size', 'sqrt', 'intra_corr'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    apply_linearization=[False], # False, True
    use_delta_method=[False], # False, True
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[True], # False, True
    strats_cols=STRATS_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [50]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:11<00:00, 88.78it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'sqrt', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:11<00:00, 88.47it/s]


Running A/A for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'intra_corr', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:11<00:00, 88.03it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,size,False,False,t_test,True,None,None,None,0.490743,True
1,sqrt,False,False,t_test,True,None,None,None,0.509854,True
2,intra_corr,False,False,t_test,True,None,None,None,0.507703,True


All weighting methods are valid.

In [51]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='ratio',
    uplift=model_uplift
)

Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'size', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:12<00:00, 79.45it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'sqrt', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:12<00:00, 79.20it/s]


Running A/B for config: {'num': 'clicks', 'den': 'views', 'weight_method': 'intra_corr', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': None, 'var_reduction_params': None, 'handle_outliers': None, 'outliers_contamination': None, 'outliers_cols': ['views_cov', 'clicks_cov', 'purchases_cov'], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:12<00:00, 78.30it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,var_reduction,handle_outliers,outliers_contamination,power
0,size,False,False,t_test,True,None,None,None,0.866000
1,sqrt,False,False,t_test,True,None,None,None,0.984000
2,intra_corr,False,False,t_test,True,None,None,None,0.935000


sqrt weighting gives the highest power (0.98), intra_corr is second (0.94), size weighting is the lowest (0.87).<br>
Keep in mind that weights change the metric itself: size weighting estimates the ratio of sums (the overall CTR), while sqrt and intra_corr give more importance to users with few views. In this simulation the effect is injected as additive noise to clicks, so it raises the CTR of users with few views the most, and the methods that weight them higher see a larger effect. Real effects are not necessarily distributed like that.

Overall, best setup is:
- t-test
- Use stratification
- Use sqrt or intra_corr weighting of users, if this metric definition suits the task

In this simulation it increases statistical power from 0.79 for the default size weighting with the delta method to 0.94–0.98. With size weighting, stratification alone gives 0.85.